# **Pompeufarrers - *Sanctions Through Statistics***

In this project, we aim to answer the following question:

***Is there a pattern behind PEPs with sanctions?***


---



For that, we have two blocks of code:
1. Downloading the OpenSanctions CSV files containing all PEPs names
2. For 11 selected countries, we extract its PEPs and match them (with filters) to OpenSanctions' sanctions database through an API.

In [ ]:
# This script downloads the latest OpenSanctions datasets for PEPs and sanctions, and generates clean CSVs for each.

import os
import csv
import requests
import pandas as pd
from bs4 import BeautifulSoup

##### SETUP #####

PATH = ""

# HTTP request timeouts, batch size for reading large CSVs, and HTTP headers
REQUEST_TIMEOUT = 120
CHUNK_SIZE_ROWS = 200_000
HEADERS = {"User-Agent": "Mozilla/5.0 (opensanctions-export/1.0)"}

# DARC URLs for sanctions and PEP datasets
DARC_DEFAULT_PAGE = "https://dataresearchcenter.org/library/default/"
DARC_PEPS_PAGE = "https://dataresearchcenter.org/library/peps/"

# Outputs
OUT_PEPS = PATH + "peps_only.csv"
OUT_SANCTIONS = PATH + "sanctions_only.csv"

# Temporarily downloaded files
DEFAULT_TARGETS_FILE = PATH + "opensanctions_default_targets.simple.csv"
PEPS_TARGETS_FILE = PATH + "opensanctions_peps_targets.simple.csv"


##### UTILITIES #####

def find_targets_simple_csv_url(page_url: str) -> str:
   """
   Finds CSV URLs automatically from DARC
   """
   r = requests.get(page_url, timeout=REQUEST_TIMEOUT, headers=HEADERS)
   r.raise_for_status()
   soup = BeautifulSoup(r.text, "html.parser")
   links = [a.get("href") for a in soup.find_all("a", href=True)]
   links = [u for u in links if u and "data.opensanctions.org" in u and "targets.simple.csv" in u]
   if not links:
       raise RuntimeError(f"targets.simple.csv not found in: {page_url}")
   return links[0] # there is usually 1

def download_file(url: str, path: str):
  """
  Downloads a file from a URL in streaming mode
  """
  if os.path.exists(path):
      print(f"{path} already exists")
      return
  print(f"Downloading: {url}")
  with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT, headers=HEADERS) as r:
      r.raise_for_status()
      with open(path, "wb") as f:
          for chunk in r.iter_content(chunk_size=1024 * 1024):
              if chunk:
                  f.write(chunk)
  print(f"Download completed: {path}")

def nonempty(series: pd.Series) -> pd.Series:
  """
  Non-empty check for pandas series
  """
  s = series.fillna("").astype(str).str.strip()
  return (s != "") & (s != "[]") & (s.str.lower() != "nan")


#### DOWNLOAD CSVS #####

# default dataset for sanctions csv
default_targets_url = find_targets_simple_csv_url(DARC_DEFAULT_PAGE)
print("✅ DEFAULT targets.simple.csv URL:", default_targets_url)
download_file(default_targets_url, DEFAULT_TARGETS_FILE)

# PEPs dataset (with fallbacks)
try:
   peps_targets_url = find_targets_simple_csv_url(DARC_PEPS_PAGE)
except Exception as e:
   print("Could not parse PEPs page automatically.")
   print("Quick solution: include peps/targets.simple.csv's URL here.")
   print("Error:", e)
   # EXAMPLE:
   peps_targets_url = "https://data.opensanctions.org/datasets/20250513/peps/targets.simple.csv?v=20250513152703-lqe"

print("PEPS targets.simple.csv URL:", peps_targets_url)
download_file(peps_targets_url, PEPS_TARGETS_FILE)


##### GENERATE SANCTIONS CSV #####

with open(OUT_SANCTIONS, "w", newline="", encoding="utf-8-sig", errors="replace") as out_f:
   w = csv.writer(out_f)
   w.writerow(["category", "id", "name", "schema", "countries", "dataset", "sanctions"])

   total = 0
   for chunk in pd.read_csv(
       DEFAULT_TARGETS_FILE,
       chunksize=CHUNK_SIZE_ROWS,
       dtype=str,
       keep_default_na=False,
       encoding_errors="replace",
   ):
       required = ["id", "name", "schema", "dataset", "sanctions"]
       missing = [c for c in required if c not in chunk.columns]
       if missing:
           raise RuntimeError(f"DEFAULT targets.simple.csv without columns {missing}. Contains: {list(chunk.columns)}")

       sanc = chunk[nonempty(chunk["sanctions"])].copy()
       if sanc.empty:
           continue

       countries = sanc["countries"] if "countries" in sanc.columns else ""
       w.writerows(zip(
           ["sanctions"] * len(sanc),
           sanc["id"],
           sanc["name"],
           sanc["schema"],
           countries,
           sanc["dataset"],
           sanc["sanctions"],
       ))

       total += len(sanc)
       print(f"Sanctions written: {total:,}")

print("DONE: ", OUT_SANCTIONS)


##### GENERATE PEPS CSV #####

with open(OUT_PEPS, "w", newline="", encoding="utf-8-sig", errors="replace") as out_f:
   w = csv.writer(out_f)
   w.writerow(["category", "id", "name", "schema", "countries", "dataset"])

   total = 0
   for chunk in pd.read_csv(
       PEPS_TARGETS_FILE,
       chunksize=CHUNK_SIZE_ROWS,
       dtype=str,
       keep_default_na=False,
       encoding_errors="replace",
   ):
       required = ["id", "name", "schema", "dataset"]
       missing = [c for c in required if c not in chunk.columns]
       if missing:
           raise RuntimeError(f"PEPS targets.simple.csv without columns {missing}. Contains: {list(chunk.columns)}")

       pep = chunk.copy()

       if pep.empty:
           continue

       countries = pep["countries"] if "countries" in pep.columns else ""
       w.writerows(zip(
           ["peps"] * len(pep),
           pep["id"],
           pep["name"],
           pep["schema"],
           countries,
           pep["dataset"],
       ))

       total += len(pep)
       print(f"peps written: {total:,}")

print("DONE: ", OUT_PEPS)


##### QUICK REPORT #####

sanction_ids = set()
for chunk in pd.read_csv(OUT_SANCTIONS, chunksize=CHUNK_SIZE_ROWS, dtype=str, encoding="utf-8-sig"):
    sanction_ids.update(chunk["id"].astype(str).str.strip().dropna())

pep_rows = 0
for chunk in pd.read_csv(OUT_PEPS, chunksize=CHUNK_SIZE_ROWS, dtype=str, encoding="utf-8-sig"):
    pep_rows += len(chunk)

print("Sanctions:", len(sanction_ids))
print("PEP:", pep_rows)


In [ ]:
# This script screens all Spanish PEPs from OpenSanctions against global sanctions lists
# using the OpenSanctions Match API and computes the percentage of PEPs with sanctions.

# The API was queried once to obtain the matching results. The returned data was then stored locally as a CSV 
# so that all subsequent analysis could be performed without making additional API calls.


import pandas as pd
import os
import requests
import json


##### SETUP #####

INPUT_PEPS = PATH + "peps_only.csv"
OUTPUT_ALL_MATCHES = PATH + "all_matches.csv"
OUTPUT_FIL_MATCHES = PATH + "filtered_matches.csv"


##### API KEY #####

API_KEY = None
API_URL = "https://api.opensanctions.org/match/sanctions"
PARAMS = {"algorithm": "best"}

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENSANCTIONS_API_KEY")
except Exception:
    pass

if not API_KEY:
    API_KEY = os.environ.get("OPENSANCTIONS_API_KEY")

if not API_KEY:
    raise RuntimeError("OPENSANCTIONS_API_KEY not found")

print("API key loaded")

session = requests.Session()
session.headers["Authorization"] = f"ApiKey {API_KEY}"


##### HELPERS #####

def bucket(score):
    """Categorizes similarity based on score."""
    if score >= 0.90: return "HIGH"
    if score >= 0.75: return "MED"
    return "LOW"

def extract_rows(responses):
    """Extract relevant fields from OpenSanctions API responses."""
    rows = []
    for qid, res in (responses or {}).items():
        for m in res.get("results", []):
            props = m.get("properties", {}) or {}
            topics = m.get("topics", []) or []
            datasets = m.get("datasets", []) or []
            score = float(m.get("score", 0) or 0)
            rows.append({
                "query_id": qid,
                "matched_name": m.get("caption"),
                "schema": m.get("schema"),
                "countries": ", ".join(props.get("country", []) or []),
                "score": score,
                "risk": bucket(score),
                "datasets": ", ".join(datasets),
                "topics": ", ".join(topics)
            })
    return rows


def make_person(name, country=None):
    """Create an OpenSanctions Person query."""
    props = {"name": [name]}
    if country:
        props["country"] = [country]
    return {"schema": "Person", "properties": props}

def build_queries(df):
    queries = {}
    for i, row in df.iterrows():
        queries[f"pep_{i}"] = make_person(row["name"], country=row["countries"])
    return queries

def run_match(queries):
    """Send all PEPs to the OpenSanctions Match API."""
    response = session.post(API_URL, json={"queries": queries}, params=PARAMS)
    if not response.ok:
        print("Error:", response.status_code)
        try:
            print(response.json())
        except Exception:
            print(response.text[:500])
        response.raise_for_status()
    return extract_rows(response.json().get("responses", {}))


##### LOAD DATA #####

df_peps = pd.read_csv(INPUT_PEPS)

queries = build_queries(df_peps)
print(f"Sending 1 single API request with {len(queries)} queries...")

all_rows = run_match(queries)


##### SAVE RESULTS #####

df_all_matches = pd.DataFrame(all_rows)
print("Total matches:", len(df_all_matches))

df_all_matches.to_csv(OUTPUT_ALL_MATCHES, index=False)
print("Saved sanctions matches to:", OUTPUT_ALL_MATCHES)


##### FILTER BY COUNTRY (example: Spain) #####

df_peps_spain = df_peps[df_peps["countries"].astype(str).str.lower().str.contains("es")].copy()
spain_query_ids = {f"pep_{i}" for i in df_peps_spain.index}
df_spain_matches = df_all_matches[df_all_matches["query_id"].isin(spain_query_ids)].copy()

df_spain_matches.to_csv(OUTPUT_FIL_MATCHES, index=False)
print(f"Filtered {len(df_spain_matches)} Spanish matches to {OUTPUT_FIL_MATCHES}")

if len(df_peps_spain) > 0:
    pct_matched = (df_all_matches["query_id"].nunique()) / len(df_peps_spain)*100
    print(f"{pct_matched:.2f}% of Spanish PEPs match sanctions")

API key loaded
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PEPs in Spain: 7150
Processing batch 1...
Processing batch 2...
Processing batch 3...
Processing batch 4...
Processing batch 5...
Processing batch 6...
Processing batch 7...
Processing batch 8...
Processing batch 9...
Processing batch 10...
Processing batch 11...
Processing batch 12...
Processing batch 13...
Processing batch 14...
Processing batch 15...
Processing batch 16...
Processing batch 17...
Processing batch 18...
Processing batch 19...
Processing batch 20...
Processing batch 21...
Processing batch 22...
Processing batch 23...
Processing batch 24...
Processing batch 25...
Processing batch 26...
Processing batch 27...
Processing batch 28...
Processing batch 29...
Processing batch 30...
Processing batch 31...
Processing batch 32...
Processing batch 33...
Processing batch 34...
Processing batch 35...
Processing batch 36...
Processing batch 